In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 04 · The runtime policy enforcement point

**Primer section:** §4 authorization patterns — four PEPs (§4.1), tool tiers and deny-by-default
(§4.2), human-in-the-loop done right (§4.4).

The layer you control most as the agent developer is the **runtime** PEP: a callback that sees the
resolved tool name and arguments *before* execution. Here it is `PolicyEngine` (a deny-by-default
evaluator over a YAML policy) wrapped by `SecurityPlugin` (an ADK plugin registered once on the
`Runner`). This notebook first evaluates requests against the policy directly, then runs the real
ADK loop — default deny, an argument envelope that waives confirmation, and the confirmation round
trip where the human approves or rejects.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

from agentsec.agents import REFUNDS, LocalStack, Step, reset_demo_state
from agentsec.audit import AuditLog
from agentsec.config import Settings
from agentsec.identity import AgentIdentity, AuthorityContext, UserPrincipal
from agentsec.policy import Effect, Policy, PolicyEngine, ToolCallRequest
from agentsec.runtime import confirm, run_turn, seed_session

reset_demo_state()

settings = Settings()
agent = settings.agent_identity()
ORG, PROJECT = settings.org_id, settings.project_number
other_project_agent = AgentIdentity.for_agent_engine(project_number="111111111111", location="us-central1", engine_id="marketing-agent", org_id=ORG)
sibling_agent = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="billing-agent", org_id=ORG)
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")

policy = Policy.from_yaml(settings.policy_path)
engine = PolicyEngine(policy)

def req(tool, authority, **args):
    return ToolCallRequest(tool=tool, args=args, authority=authority)

def show(label, decision):
    print(f"{label:<58} {decision.effect.value.upper():<8} {'; '.join(decision.reasons) or '-'}")

## 1. Policy is data, reviewed like IAM

`policies/support-agent.yaml` classifies every tool once: tier (read / write / destructive /
external), which principals may call it, the authority mode, required scopes, argument
constraints, confirmation rules, egress hosts. Unknown tool → deny. `default: allow` is rejected by
the model on purpose.

In [ ]:
print("default:", policy.default, "| budgets:", policy.budgets.model_dump())
print(f"\n{'tool':<22} {'tier':<12} {'allow':<16} {'authority':<10} {'scopes':<18} confirmation")
for name, tp in policy.tools.items():
    conf = "required" if tp.confirmation.required else "-"
    if tp.confirmation.unless:
        conf += f" unless {tp.confirmation.unless}"
    print(f"{name:<22} {tp.tier.value:<12} {','.join(tp.allow):<16} {tp.authority.value:<10} {','.join(tp.required_scopes) or '-':<18} {conf}")

from pydantic import ValidationError

try:
    Policy.from_dict({"default": "allow", "tools": {}})
except ValidationError as e:
    print("\nallow-by-default refused:", e.errors()[0]["msg"])

## 2. Evaluate requests: the deny reasons, one at a time

Evaluation order: tool known → principal allowed → authority mode → scopes → constraints → egress
→ budgets → confirmation. Every failure becomes a *reason* the audit log and the model can see.

In [ ]:
delegated = AuthorityContext.delegated(agent, ana, {"customers:read", "orders:read", "payments:refund", "email:send"})
own = AuthorityContext.own(agent, {"customers:read"})

show("run_sql (not in policy)", engine.evaluate(req("run_sql", delegated, query="select 1")))
show("lookup_customer as marketing-agent (other project)", engine.evaluate(req("lookup_customer", AuthorityContext.delegated(other_project_agent, ana, {"customers:read"}), email="x")))
show("issue_refund as billing-agent (same project, not named)", engine.evaluate(req("issue_refund", AuthorityContext.delegated(sibling_agent, ana, {"payments:refund"}), order_id="O-5002", amount=10, currency="USD", reason="x")))
show("lookup_customer under OWN authority", engine.evaluate(req("lookup_customer", own, email="x")))
show("lookup_customer delegated but no scope", engine.evaluate(req("lookup_customer", AuthorityContext.delegated(agent, ana, set()), email="x")))
show("search_knowledge under OWN authority (authority: any)", engine.evaluate(req("search_knowledge", own, query="refund")))
show("lookup_customer delegated with scope", engine.evaluate(req("lookup_customer", delegated, email="ana@customer.example")))
assert engine.evaluate(req("run_sql", delegated, query="select 1")).effect is Effect.DENY

## 3. Constraints, confirmation, and the `unless` envelope

`issue_refund` is destructive: amount within `[0.01, 500]`, currency in `{USD, SGD}`, and
confirmation required **unless** `args.amount <= 50 and args.currency == 'USD'`. The `unless`
expression is a tiny safe language (like CEL in IAM Conditions). A request a human already approved
(`confirmed_by`) is allowed and the approver becomes part of the record.

In [ ]:
def refund(**a):
    return req("issue_refund", delegated, **{"order_id": "O-5001", "currency": "USD", "reason": "duplicate charge", **a})

show("refund 35 USD  (inside envelope)", engine.evaluate(refund(amount=35)))
show("refund 120 USD (outside envelope)", engine.evaluate(refund(amount=120)))
show("refund 35 SGD  (envelope is USD-only)", engine.evaluate(refund(amount=35, currency="SGD")))
show("refund 5000 USD (constraint max)", engine.evaluate(refund(amount=5000)))
show("refund 10 EUR  (constraint enum)", engine.evaluate(refund(amount=10, currency="EUR")))
approved = ToolCallRequest(tool="issue_refund", args={"order_id": "O-5001", "amount": 120, "currency": "USD", "reason": "x"}, authority=delegated, confirmed_by="ana@customer.example")
show("refund 120 USD with confirmed_by", engine.evaluate(approved))
d = engine.evaluate(refund(amount=120))
print("\nconfirmation hint shown to the human:", d.hint)

## 4. Egress and budgets

Any tool that takes a URL gets an egress allowlist (parsed host, HTTPS only, no private ranges —
SSRF and exfiltration are the same bug in an agent). `send_email` constrains the recipient domain by
pattern. Budgets cap tool calls and destructive calls per invocation; `dry_run` evaluates a whole
plan with budget accounting (plan → check → act).

In [ ]:
for url in ["https://docs.acme.example/refunds", "https://storage.googleapis.com/bucket/x", "http://docs.acme.example/x",
            "https://evil.example/x", "https://169.254.169.254/computeMetadata/v1/", "https://docs.acme.example.evil.example/", "https://user:pw@docs.acme.example/"]:
    show(f"fetch_url {url}", engine.evaluate(req("fetch_url", own, url=url)))

show("send_email to attacker@evil.example", engine.evaluate(req("send_email", delegated, to="attacker@evil.example", subject="s", body="b")))
show("send_email to ana@customer.example", engine.evaluate(req("send_email", delegated, to="ana@customer.example", subject="s", body="b")))

over_budget = ToolCallRequest(tool="lookup_customer", args={"email": "x"}, authority=delegated, counters={"tool_calls": 12})
show("lookup_customer as the 13th call", engine.evaluate(over_budget))

In [ ]:
plan = [
    ("lookup_customer", {"email": "ana@customer.example"}),
    ("issue_refund", {"order_id": "O-5001", "amount": 10, "currency": "USD", "reason": "a"}),
    ("issue_refund", {"order_id": "O-5002", "amount": 10, "currency": "USD", "reason": "b"}),
    ("issue_refund", {"order_id": "O-5003", "amount": 10, "currency": "USD", "reason": "c"}),
    ("send_email", {"to": "ana@customer.example", "subject": "done", "body": "…"}),
]
print("dry run of the plan:")
for (tool, args), d in zip(plan, engine.dry_run(plan, delegated), strict=True):
    show(f"  {tool}({args.get('order_id') or args.get('email') or args.get('to')})", d)

## 5. The same policy inside the ADK loop

`LocalStack` wires the policy engine into `SecurityPlugin.before_tool_callback`. The scripted model
tries four things in one turn: a read (allowed), `run_sql` (not in policy → the callback returns an
error dict and the tool never runs), a refund inside the envelope (auto-allowed), and a refund outside
it (paused with `adk_request_confirmation`). The front-end's job — modelled by `runtime.py` — is to
seed the session with the verified user and scopes, and to render the confirmation with the **actual**
tool call.

In [ ]:
stack = LocalStack.create(Settings(), audit=AuditLog())
USER = {"subject": "u-ana", "email": "ana@customer.example", "tenant": "acme"}
SCOPES = ["customers:read", "orders:read", "payments:refund", "email:send"]
await seed_session(stack.runner, user_id="u-ana", session_id="s1", user=USER, scopes=SCOPES)

stack.script(
    Step.call("lookup_customer", email="ana@customer.example"),
    Step.call("run_sql", query="select * from customers"),
    Step.call("issue_refund", order_id="O-5002", amount=35.0, currency="USD", reason="duplicate charge"),
    Step.call("issue_refund", order_id="O-5001", amount=120.0, currency="USD", reason="event cancelled"),
    Step.say("I refunded O-5002 and I need your approval for O-5001."),
)
r = await run_turn(stack.runner, user_id="u-ana", session_id="s1", message="please refund my orders")
print(r.summary())

by_name = {t["name"]: t["response"] for t in r.tool_responses}
assert "customer" in by_name["lookup_customer"]
assert by_name["run_sql"]["error"] == "policy_denied"
assert len(REFUNDS) == 1 and REFUNDS[0]["amount"] == 35.0
assert len(r.pending_confirmations) == 1

### The confirmation the human sees is the real call, not the model's prose

The payload carries the tool, the (redacted) arguments, the authority (who is acting for whom), the
tier and the policy reasons. That is what a confirmation UI must render (primer §4.4, drill 4).

In [ ]:
p = r.pending_confirmations[0]
print("original call:", p.original_call["name"], p.original_call["args"])
print("hint         :", p.hint)
print("payload      :", {k: p.payload[k] for k in ("tool", "args", "authority", "tier")})

stack.script(Step.say("Refund of 120 USD on O-5001 issued."))   # what the model says after the tool result
r_ok = await confirm(stack.runner, user_id="u-ana", session_id="s1", pending=p, confirmed=True)
print("\nafter approval:\n" + r_ok.summary())
assert any(t["name"] == "issue_refund" and t["response"].get("status") == "issued" for t in r_ok.tool_responses)
assert len(REFUNDS) == 2 and REFUNDS[-1]["amount"] == 120.0

### Rejecting the confirmation executes nothing

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="s2", user=USER, scopes=SCOPES)
stack.script(Step.call("issue_refund", order_id="O-5003", amount=480.0, currency="SGD", reason="customer request"), Step.say("ok"))
r2 = await run_turn(stack.runner, user_id="u-ana", session_id="s2", message="refund O-5003")
p2 = r2.pending_confirmations[0]
stack.script(Step.say("Understood — nothing was refunded."))
r2_no = await confirm(stack.runner, user_id="u-ana", session_id="s2", pending=p2, confirmed=False)
print(r2_no.summary())
assert any(t["response"].get("error") == "policy_denied" for t in r2_no.tool_responses)
assert len(REFUNDS) == 2  # unchanged

## 6. The audit trail the plugin left behind

One event per decision, with both identities, the decision and its reasons, and the approver when a
human confirmed.

In [ ]:
for row in stack.audit.timeline():
    print(row)
approvals = [e for e in stack.audit.events() if e.approver]
assert approvals and approvals[0].approver == "ana@customer.example"

## The four PEPs on Google Cloud

1. **IAM on the resource** — the final arbiter: allow bindings on the agent principal, IAM Conditions,
   deny policies and Principal Access Boundaries for "never, regardless".
2. **Runtime (this notebook)** — ADK `before_tool_callback` / plugin: allowlist, tiers, argument
   constraints, scopes, budgets, confirmation. Returning a result from the callback skips the tool.
3. **Network** — Agent Gateway as the single egress point; IAP per SPIFFE ID; VPC-SC ingress/egress
   rules with `mcp.toolName` / `mcp.tool.isReadOnly` conditions.
4. **Model-side** — Model Armor floor settings and templates (notebook 05).

**In one sentence:** "I classify tools into read, write and destructive in a policy file
reviewed like IAM, and enforce it in the runtime callback that sees the real tool call: unknown tools
are denied, destructive tools need confirmation unless a narrow argument envelope holds, and the
confirmation shows the actual arguments. IAM and the perimeter still hold if my code is wrong."